# Agent webUI

Let's build a web interface to test the agent. The interface will be available via browser by proxying the port through the workspace.

## Project Initialization

Initialize a DigitalHub project using consistent naming with other tutorials.


In [ ]:
import digitalhub as dh
import getpass as gt

USERNAME = gt.getuser()

project = dh.get_or_create_project(f"{USERNAME}-rag")
print(project.name)

## 1. Fetch components 

Let's get the components definition. We are looking for the latest RUNNING run for the agent

In [ ]:
rag_function = project.get_function("rag-service")
rag_run = rag_function.list_runs()[0]
if rag_run:
    print(f"Found run {rag_run.id}")

In [ ]:
AGENT_URL = rag_run.status.to_dict()["service"]["url"]
print(f"service {AGENT_URL} found")


## Build the UI

We use Gradio to serve a simple webpage with an input field connected to the agent API.

[Gradio](https://gradio.app/) is a Python framework to create browser applications with little code.

In [ ]:
%pip install -qU gradio==6.12.0 dotenv

Write the function implementing the RAG ui to file

In [ ]:
import gradio as gr
import requests


def stream_response(message, history):
    #TODO history to enable memory
    # print("History:", history)

    try:
        # Call the agent API
        res = requests.post(f"http://{AGENT_URL}", json={"question": message})
        response = res.json()
        print(response)

        return response.get("answer", "No response from agent")
    except Exception as e:
        print("Error:", e)
        return "Error occurred while processing the request"

chatbot = gr.ChatInterface(stream_response).queue()

In [ ]:
chatbot.launch(debug=True, share=True)


## Excercises

* Package the application as container and run as container-serve
* refactor serve to support history and add to frontend
* refactor serve to expose openai protocol